In [6]:
pip install sqlalchemy

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 5.6 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.2 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 4.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import holidays
from sqlalchemy import create_engine, text
import psycopg2
import os

In [8]:
country_codes = [
    "ET",  # Ethiopia
    "ER",  # Eritrea
    "SV",  # El Salvador
    "CN",  # China
    "IN",  # India
    "PK",  # Pakistan
    "BD",  # Bangladesh
    "VN",  # Vietnam
    "US",  # United States
    "CA"   # Canada
]

country_code_replacements = {
                    "ET":"Ethiopia",
                    "ER":"Eritrea",
                    "SV":"El Salvador",
                    "CN":"China",
                    "IN":"India",
                    "PK":"Pakistan",
                    "BD":"Bangladesh",
                    "VN":"Vietnam",
                    "US":"United  States",
                    "CA":"Canada"
                }

In [50]:
conn = psycopg2.connect(
    host="localhost",
    port=5433,
    user=os.environ["WAREHOUSE_USER"],
    password=os.environ["WAREHOUSE_PASSWORD"],
    dbname="bikeshare",
)
cur = conn.cursor()

engine = create_engine(
    f"postgresql+psycopg2://{os.environ['WAREHOUSE_USER']}:{os.environ['WAREHOUSE_PASSWORD']}"
    f"@localhost:5433/bikeshare"
)

In [27]:
cur.execute("SELECT DISTINCT(start_time::date) AS Date FROM silver.trips;")
dim_date = pd.DataFrame(cur.fetchall(), columns = ["Date"])

In [29]:
all_holidays = []

for code in country_codes:
    h = holidays.CountryHoliday(code, years=2020)
    for date, name in h.items():
        all_holidays.append({"country": code, "holiday_date": date, "holiday_name": name})
holiday_list = pd.DataFrame(all_holidays)

C:\Users\Darshil\AppData\Local\Temp\ipykernel_15344\3209154753.py:4: DeprecationWarning: CountryHoliday is deprecated, use country_holidays instead.
  h = holidays.CountryHoliday(code, years=2020)


In [30]:
holiday_list["holiday_date"] = pd.to_datetime(holiday_list["holiday_date"])

In [31]:
holiday_list["country"] = holiday_list["country"].replace(country_code_replacements)

In [32]:
countries = holiday_list.groupby("holiday_date")["country"].apply(list).reset_index()
holidays = holiday_list.groupby("holiday_date")["holiday_name"].apply(list).reset_index()

In [33]:
final = pd.merge(countries, holidays, on = "holiday_date")
final["holiday_count"] = final["country"].apply(len)
final["is_holiday"] = 1

In [34]:
final = pd.merge(left = dim_date, right = final, how = "left", left_on = "Date", right_on = "holiday_date")

ValueError: You are trying to merge on object and datetime64[s] columns for key 'Date'. If you wish to proceed you should use pd.concat

In [35]:
final

,holiday_date,country,holiday_name,holiday_count,is_holiday
0,2020-01-01,"[Eritrea, El Salvador, China, Vietnam, United ...","[New Year's Day, Año Nuevo, 元旦, Tết Dương lịch...",6,1
1,2020-01-07,"[Ethiopia, Eritrea]","[የገና ወይም የልደት በዓል, Orthodox Christmas]",2,1
2,2020-01-20,"[Ethiopia, Eritrea, United States]","[የጥምቀት በዓል, Epiphany, Martin Luther King Jr. Day]",3,1
3,2020-01-23,[Vietnam],[29 Tết],1,1
4,2020-01-24,"[China, Vietnam]","[休息日（由 2020-01-19 调休）, Giao thừa Tết Nguyên Đán]",2,1
...,...,...,...,...,...
82,2020-11-14,[India],[Diwali (Deepavali)],1,1
83,2020-11-26,[United States],[Thanksgiving Day],1,1
84,2020-11-30,[India],[Guru Nanak's Jayanti],1,1
85,2020-12-16,[Bangladesh],[বিজয় দিবস],1,1


In [37]:
final = final[["holiday_date", "country", "holiday_name", "holiday_count", "is_holiday"]]

In [39]:
final["day"] = final["holiday_date"].dt.day_name()

In [40]:
long_weekends = final.loc[final["day"].isin(["Friday", "Monday"])].index
final["long_weekend"] = 0
final.loc[long_weekends, "long_weekend"] = 1

In [41]:
longer_weekends = final.loc[final["day"].isin(["Tuesday", "Thursday"])].index
final["mini_vac_blocker"] = 0
final.loc[longer_weekends, "mini_vac_blocker"] = 1

In [42]:
final["is_holiday"] = final["is_holiday"].fillna(0, inplace = True)
final["holiday_count"] = final["holiday_count"].fillna(0, inplace = True)

C:\Users\Darshil\AppData\Local\Temp\ipykernel_15344\1220374192.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  final["is_holiday"] = final["is_holiday"].fillna(0, inplace = True)
C:\Users\Darshil\AppData\Local\Temp\ipykernel_15344\1220374192.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series thr

In [43]:
final["holiday_count"] = final["holiday_count"].astype(int)
final["is_holiday"] = final["is_holiday"].astype(int)

In [44]:
final.loc[final["day"].isin(["Saturday", "Sunday"]), "is_holiday"] = 1

In [45]:
final["country"] = final["country"].fillna("none")
final["holiday_name"] = final["holiday_name"].fillna("none")

In [46]:
final.columns = ["Date", "Country", "Holiday", "Holiday_count", "Is_holiday", "Day", "Long_weekend", "Mini_vac_blocker"]
final = final[["Date", "Day", "Is_holiday", "Holiday_count", "Long_weekend", "Mini_vac_blocker", "Holiday", "Country"]]

In [47]:
final

,Date,Day,Is_holiday,Holiday_count,Long_weekend,Mini_vac_blocker,Holiday,Country
0,2020-01-01,Wednesday,1,6,0,0,"[New Year's Day, Año Nuevo, 元旦, Tết Dương lịch...","[Eritrea, El Salvador, China, Vietnam, United ..."
1,2020-01-07,Tuesday,1,2,0,1,"[የገና ወይም የልደት በዓል, Orthodox Christmas]","[Ethiopia, Eritrea]"
2,2020-01-20,Monday,1,3,1,0,"[የጥምቀት በዓል, Epiphany, Martin Luther King Jr. Day]","[Ethiopia, Eritrea, United States]"
3,2020-01-23,Thursday,1,1,0,1,[29 Tết],[Vietnam]
4,2020-01-24,Friday,1,2,1,0,"[休息日（由 2020-01-19 调休）, Giao thừa Tết Nguyên Đán]","[China, Vietnam]"
...,...,...,...,...,...,...,...,...
82,2020-11-14,Saturday,1,1,0,0,[Diwali (Deepavali)],[India]
83,2020-11-26,Thursday,1,1,0,1,[Thanksgiving Day],[United States]
84,2020-11-30,Monday,1,1,1,0,[Guru Nanak's Jayanti],[India]
85,2020-12-16,Wednesday,1,1,0,0,[বিজয় দিবস],[Bangladesh]


In [51]:
with engine.begin() as c:
    c.execute(text('CREATE SCHEMA IF NOT EXISTS gold'))

final.to_sql("DIM_DATE", engine, schema="gold", if_exists="replace", index=False)

87